# Molecular diffusion on the asym50_95 N=250k networks

Steady-state molecular diffusion (Fick + Knudsen combined, `diff_model="All"`) solved in the three
directions x/y/z on one of the four 250k-pore networks generated by `generate_diffusion_test_networks.py`:
four independent realizations (seeds) of the single best asym50_95 recipe (window [0.068, 26.53] um — see
`doc/experiment-summary.typ`).

For each direction the effective diffusivity Deff is reported (in m^2/s and as Deff/Dinf) and saved
to a CSV file in the current directory. Run this notebook from the `examples/` folder: networks are
read from `examples/networks/`.


In [ ]:
import networkx as nx
import numpy as np
import pnm
import matplotlib as mpl
import matplotlib.pyplot as plt
import os
import csv
import time
print(nx.__version__)


In [ ]:
# One of the four asym50_95 N=250k networks generated by generate_diffusion_test_networks.py:
# four independent realizations (seeds) of the single best recipe — kernel e=0.75/sratio=0.7,
# v1 two-phase "surf" order="decreasing", phase1_z=0, p2f=2, maxc=1024 (JSD 0.0093 @ N=150k).
netfile = "20261120_asym5095_250k_kernel_e0.75_sr0.7_dec_seed7.gml"
# netfile = "20261120_asym5095_250k_kernel_e0.75_sr0.7_dec_seed8.gml"
# netfile = "20261120_asym5095_250k_kernel_e0.75_sr0.7_dec_seed9.gml"
# netfile = "20261120_asym5095_250k_kernel_e0.75_sr0.7_dec_seed10.gml"

folder = os.path.join(os.getcwd(), "networks")
print('Loading the network...',end="")
pn  = pnm.pore_network.read(os.path.join(folder, netfile), method="gml")
print("OK")
print("{} pores and {} throats".format(pn.graph.number_of_nodes(),pn.graph.number_of_edges()))


Note : d'après T. Sanchez le coeff effectif relatif de diffusion moléculaire pour des pâtes de type CEM-V est de l'ordre de 2.6e-4 à 2.8e-4. Ca parait assez faible / coeff diff sur des mortiers ?

In [3]:
print('Setting up the simulation...',end="")
scale = 1e-6
#pnm.scale(pn,scale)
BC1 = 1
BC2 = 0
Dinf = 1.25e-5
Ma = 44e-3 #Masse atomique en kg/mol
DKNUDSEN = (1/6)*np.sqrt((8*8.3144*293)/(np.pi*Ma))
S = {'x':np.prod(pn.graph.graph['extent'][1:]),
    'y':np.prod(pn.graph.graph['extent'][0::2]),
    'z':np.prod(pn.graph.graph['extent'][:2])}
L = {'x':pn.graph.graph['extent'][0],
    'y':pn.graph.graph['extent'][1],
    'z':pn.graph.graph['extent'][2]}
Deff = {}

throat_model = "throat"
diff_model = "All" #"Fick"
print("OK")

nbcpores = {}
for direction in ['x','y','z']:
    dir1 = direction + "+"
    dir2 = direction + "-"
    nbcpores[direction] = 100*(len(pn.get_pores_by_category([dir1])) +
            len(pn.get_pores_by_category([dir2])) ) / pn.graph.number_of_nodes()
    print(nbcpores[direction])

Setting up the simulation...OK
0.4
0.4
0.4


In [ ]:
# One direction for now; extend to all three as needed:
directions = ['x']   # e.g. ['x', 'y', 'z']
for direction in directions:

    dir1 = direction + "+"
    dir2 = direction + "-"

#Note that when throat radius is "effective" the sum of pore throats surface can be greater than the network surface...
#which can lead to Deff>Dinf
    t0 = time.perf_counter()
    solver = pnm.molecular_diffusion(pn)
    solver.setup_throat_model(model=throat_model)
    solver.setup_diffusivity_model(model=diff_model,d_fick=Dinf,d_knudsen = DKNUDSEN, scale = 1/scale)
    solver.add_bc(BC1,pn.get_pores_by_category([dir1],mode='match_one'),bc_type='Dirichlet')
    solver.add_bc(BC2,pn.get_pores_by_category([dir2],mode='match_one'),bc_type='Dirichlet')
    print('Building coeff matrix...',end="")
    solver.setup_coef_matrix()
    print("OK")
    print("Solving...",end="")
    solver.solve(solver='cg',tol=1e-8)
    print('OK (status {})'.format(solver.iter))
    q1 = solver.compute_boundary_flow(pn.get_pores_by_category([dir1],mode='match_one'))
    q2 = solver.compute_boundary_flow(pn.get_pores_by_category([dir2],mode='match_one'))
    print("Flow in", -q1," Flow out:",q2)
    Deff[direction] = q1*L[direction]*scale**2 /((BC2-BC1)*S[direction])
    print('Deff {} = {:.3e}, Deff/Dinf = {:.3e}'.format(direction,Deff[direction] ,Deff[direction]/Dinf))
    print("Direction {}: total {:.0f}s".format(direction, time.perf_counter()-t0))

    #nbcporeslabels = pn.get_pores_by_category([dir2]).extend(pn.get_pores_by_category([dir1]))
    #nbcpores[direction] = sum([pn.graph.nodes[n]['volume'] for n in nbcporeslabels]) / pn.pores_volume
    #nbcpores[direction] = 100*(len(pn.get_pores_by_category([dir1])) +
    #        len(pn.get_pores_by_category([dir2])) ) / pn.graph.number_of_nodes()


In [6]:
name = throat_model + "-" + diff_model + "-" + os.path.split(netfile)[-1][:-4]
avgc = sum([pn.graph.degree(n) for n in pn.graph.nodes])/pn.graph.number_of_nodes()

with open(name+".csv",'w') as csvfile:
    csvwriter = csv.writer(csvfile, delimiter=';', dialect='excel')
    csvwriter.writerow(['name','nnodes','nthroats','porosity','avg connectivity','throat model',
                        '%BCP-x','%BCP-y','%BCP-z','Deff-x','Deff-y','Deff-z'])
    csvwriter.writerow([name,
                        pn.graph.number_of_nodes(),
                        pn.graph.number_of_edges(),
                        pn.graph.graph['porosity'],
                        avgc,throat_model,
                        nbcpores['x'],nbcpores['y'],nbcpores['z'],
                        Deff.get('x',''),Deff.get('y',''),Deff.get('z','')])

## Optional: 3D rendering of the concentration field

`PNM_3d_viewer` builds one VTK actor per pore and per throat — very slow and memory-hungry at
N=250k / >1M throats (its docstring recommends `Fast3DViewer` beyond a few hundred pores). The
rendering is therefore disabled by default; set `render_3d = True` on an interactive machine to save
an offscreen JPEG of the field from the last solved direction.


In [ ]:
# Optional 3D rendering of the concentration field (last simulated direction).
render_3d = False

if render_3d:
    c = {n: float(solver.x[n]) for n in pn.graph.nodes}
    norm = mpl.colors.Normalize(vmin=BC2, vmax=BC1)
    pmap = {k: np.array(mpl.cm.jet(norm(c[k]))[:-1]) for k in c}
    cmap = [0.5*(pmap[n2]+pmap[n1]) for n1,n2 in pn.graph.edges]

    viewer = pnm.PNM_3d_viewer(pn)
    viewer.make_geometry()
    # highlight the boundary pores of the last simulated direction in red
    d = directions[-1]
    pmap.update({n: [1,0,0] for n in pn.get_pores_by_category([d+'+', d+'-'])})
    viewer.maps_colors(pores_colormap = pmap, throats_colormap=cmap)
    viewer.save("diffusion_" + os.path.splitext(netfile)[0] + ".jpg")
